# ESOL Solubility Prediction with a Pretrained Molecular Language Model
We revisit a former acquaintance: The ESOL dataset (assignment 1B), but now, instead of creating our own features, we use a pretrained foundation model for encoding and add a one-layer regressor on top of it to predict the solubility.

### Preparation:
To load the pretrained transformer, the `transformers` package is needed. In order not to mess up our (in some cases quite fragile) environment, I suggest the following procedure:

1) In VSC code, after starting the environment, run: ``uv remove transformers –-optional transformers``
2) Then, run: ``uv add transformers --optional tranformers`` and ``uv sync --extra transformers``
3) Tell me the version number (need to specify the lowest in the project.toml in the repo).


### Exercise


Note: The following code has been adapted from generated snippets provided by Copilot.

1) Imports

In [1]:
# Core import: Pytorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Transformers import for loading the foundation model later
from transformers import AutoTokenizer, AutoModel

# Data & ML utilities
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Reproducibility: Random seeds
torch.manual_seed(42)
np.random.seed(42)

2) Load dataset and remove any rows with missing values (just in case). 

In [32]:
df_full = pd.read_csv("esol.csv")

# drop rows just in case either smiles or logS values are missing. 
# It is crucial to have complete and labelled data for our exercise!
df_full.dropna(axis=0, inplace=True)

# use either the full dataset or a much smaller one by sampling
# df = df_full
df = df_full.sample(20, random_state=42)
print(df.head())
print(f"Dataset size: {len(df)}")

                                   smiles   logS
1091                             CC/C=C\C -2.540
898        O=C1NC(=O)NC(=O)C1(CC)CC=C(C)C -2.253
739      Cc1[nH]c(=O)n(c(=O)c1Cl)C(C)(C)C -2.484
140                              CC/C=C/C -2.540
1019  ClC(Cl)C(c1ccc(Cl)cc1)c2ccc(Cl)cc2  -7.200
Dataset size: 20


3) Train test split (standard scikit procedure)

In [33]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

4) Load Pretrained Molecular Language Model (ChemBERTa)

We use ``ChemBERTa‑zinc‑base``, an LLM trained on millions of SMILES. We will use it first as pure transfer learning, no fine-tuning!

In [34]:
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME)

# no weights of the pretrained model are updated, used as fixed backbone
encoder.eval()
for param in encoder.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


5) SMILES embedding:

We extract a single embedding per molecule using mean pooling.

In [35]:
@torch.no_grad()
def embed_smiles(smiles_list, batch_size=32):
    all_embeddings = []

    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        outputs = encoder(**inputs)
        hidden = outputs.last_hidden_state           # (B, L, D)
        mask = inputs["attention_mask"].unsqueeze(-1)

        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        all_embeddings.append(pooled)

    return torch.cat(all_embeddings)

6) Precompute embeddings for the SMILES in our df - using the fixed encoder

In [36]:
X_train = embed_smiles(train_df["smiles"].tolist())
y_train = torch.tensor(train_df["logS"].values).float()

X_val = embed_smiles(val_df["smiles"].tolist())
y_val = torch.tensor(val_df["logS"].values).float()

print("Train embeddings:", X_train.shape)
print("Validation embeddings:", X_val.shape)

Train embeddings: torch.Size([16, 768])
Validation embeddings: torch.Size([4, 768])


7) Define a linear regression head ("Linear Probe")

We build a minimal regressor on top of our frozen embeddings. Technically, you could build anything here.

In [37]:
# minimal Pytorch regressor with one layer
class Regressor(nn.Module):
    def __init__(self, input_dim):
        super(Regressor, self).__init__()
        self.regressor = nn.Linear(input_dim, 1)

    def forward(self, x):
        x = self.regressor(x).squeeze(-1)
        return x

In [38]:
# initialise the model, define loss function and the gradient optimiser
model = Regressor(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

8) Training and evaluation functions:

In [39]:
def train_epoch(X, y, model, optimizer):
    model.train()
    optimizer.zero_grad()

    preds = model(X)
    loss = criterion(preds, y)
    loss.backward()
    optimizer.step()

    return loss.item()

A note on generated code: Always check the output! Even if the code runs, there might be caveats that would be a no-go in production!

In [40]:
def evaluate(X, y, model):
    model.eval()
    with torch.no_grad():
        preds = model(X)
        val_loss = criterion(preds, y)
# here is a classic example of generated code: this code would still run, 
# but is not sustainable (deprecated parameters)

    # rmse = mean_squared_error(y, preds, squared=False)

# recommended solution: use another metric from scikit learn (root_mean_squared_error) 
# ...or a workaround if you are too lazy to redefine the dependencies ("hacky solution"):
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2 = r2_score(y, preds)

    return val_loss, rmse, r2

9) Run training and evaluate.

Note: If you rerun this cell, you continue the training!

In [41]:
for epoch in range(300):
    loss = train_epoch(X_train, y_train, model, optimizer)
    val_loss, rmse, r2 = evaluate(X_val, y_val, model)

    if epoch % 20 == 0:
        print(
            f"Epoch {epoch:02d} | "
            # Compare the training and validation loss to check for overfitting
            f"Train loss (MSE): {loss:.3f} | "
            f"Val loss (MSE): {val_loss:.3f} | "

            # Calculate metrics to compare with other models
            f"Val RMSE: {rmse:.3f} | "
            f"Val R2: {r2:.3f}"
        )


Epoch 00 | Train loss (MSE): 13.180 | Val loss (MSE): 17.474 | Val RMSE: 4.180 | Val R2: -1.490
Epoch 20 | Train loss (MSE): 2.259 | Val loss (MSE): 5.482 | Val RMSE: 2.341 | Val R2: 0.219
Epoch 40 | Train loss (MSE): 0.463 | Val loss (MSE): 9.701 | Val RMSE: 3.115 | Val R2: -0.383
Epoch 60 | Train loss (MSE): 0.125 | Val loss (MSE): 11.557 | Val RMSE: 3.400 | Val R2: -0.647
Epoch 80 | Train loss (MSE): 0.071 | Val loss (MSE): 12.992 | Val RMSE: 3.604 | Val R2: -0.852
Epoch 100 | Train loss (MSE): 0.048 | Val loss (MSE): 13.088 | Val RMSE: 3.618 | Val R2: -0.865
Epoch 120 | Train loss (MSE): 0.032 | Val loss (MSE): 13.120 | Val RMSE: 3.622 | Val R2: -0.870
Epoch 140 | Train loss (MSE): 0.021 | Val loss (MSE): 13.034 | Val RMSE: 3.610 | Val R2: -0.858
Epoch 160 | Train loss (MSE): 0.013 | Val loss (MSE): 13.005 | Val RMSE: 3.606 | Val R2: -0.854
Epoch 180 | Train loss (MSE): 0.008 | Val loss (MSE): 12.980 | Val RMSE: 3.603 | Val R2: -0.850
Epoch 200 | Train loss (MSE): 0.005 | Val loss 

10) Compare with a classic ML model

 Check against:
 - a gradient boosting model from assignment 1B. For convenience, the joblib export was reused to use the hyperparameters from the assignment. 
 - a linear regression model
 
 Try as training data:
 - Morgan fingerprints
 - Physicochemical descriptors
 Descriptor and fingerprint generation is done in `esol_descriptors.py` to keep the notebook clean.

In [42]:
from esol_descriptors import generate_descriptors_sample, generate_mfp_sample

# use samplesize = 0 in order to load the full df
ML_X_train, ML_X_test, ML_y_train, ML_y_test = generate_mfp_sample(df_full, 20)
print(len(ML_X_train))


16


In [44]:
import joblib

gb_model = joblib.load("best_model.joblib")

gb_model.fit(ML_X_train, ML_y_train)

y_train_pred = gb_model.predict(ML_X_train)
y_test_pred = gb_model.predict(ML_X_test)

train_rmse = np.sqrt(mean_squared_error(ML_y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(ML_y_test, y_test_pred))

train_r2 = r2_score(ML_y_train, y_train_pred)
test_r2 = r2_score(ML_y_test, y_test_pred)

print("Tuned Gradient Boosting")
print(f"Train RMSE: {train_rmse:.3f}")
print(f"Test  RMSE: {test_rmse:.3f}")
print(f"Train R²: {train_r2:.3f}")
print(f"Test  R²: {test_r2:.3f}")

Tuned Gradient Boosting
Train RMSE: 0.001
Test  RMSE: 2.979
Train R²: 1.000
Test  R²: -0.265


In [18]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(ML_X_train, ML_y_train)

y_train_pred = lin_reg.predict(ML_X_train)
y_test_pred = lin_reg.predict(ML_X_test)

train_rmse = np.sqrt(mean_squared_error(ML_y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(ML_y_test, y_test_pred))

train_r2 = r2_score(ML_y_train, y_train_pred)
test_r2 = r2_score(ML_y_test, y_test_pred)

print("Linear Regression evaluation")
print(f"Train RMSE: {train_rmse:.3f}")
print(f"Test  RMSE: {test_rmse:.3f}")
print(f"Train R²: {train_r2:.3f}")
print(f"Test  R²: {test_r2:.3f}")


TypeError: float() argument must be a string or a real number, not 'Mol'

### More tasks
1) Rerun the NN pipeline using a small sample size (use ``random_randomstate=42`` to be consistent with the sampling in the smaller models (point 10). Compare and discuss.
2) Discuss the results of the small ML models for fingerprints and descriptors.
3) Discuss the performance differences of both approaches. What could be done to improve?
4) What is your take-away from the pretrained model? 